# Fine-Tuning TinyBERT on SST-2

I fine-tune TinyBERT on SST-2, then compare it against a frozen sentence-transformer and logistic regression baseline.

Based on my Master of Applied Data Science coursework.

## Part 1: Fine-tuning an Encoder-only Transformer

Fine-tuning [TinyBERT](https://huggingface.co/huawei-noah/TinyBERT_General_4L_312D) on SST-2. 

You can find a example tutorial for loading BERT and fine-tuning [here](https://huggingface.co/docs/transformers/training). 

You can use this tutorial as a guide, but if following this code you will need to make some changes throughout to work with the SST-2 dataset and only the two classes it contains.


# Library Imports

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate



In [5]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.10.0+cu126
GPU available: True


# Data Load

In [7]:
dataset = load_dataset("stanfordnlp/sst2")
print(dataset['train'][0])


{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}


# Tokenize

In [9]:
tokenizer = AutoTokenizer.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")

Error Code
```
OSError: huawei_noah/TinyBERT_General_4L_312D is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`
```

Gotcha: The model ID needs the exact HuggingFace path 'huawei-noah/TinyBERT_General_4L_312D' . Typos will throw errors and prevent progression.

In [11]:
def tokenize(batch):
    return tokenizer(batch['sentence'], padding='max_length', max_length=128, truncation=True)
    #    return tokenizer(batch['sentence'], padding=True, truncation=True)

In [12]:
td = dataset.map(tokenize, batched=True)
td = td.rename_column("label", "labels")    #Apparently there is a missing letter here?
td.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

# Results

In [14]:
print(td['train'][0])

{'labels': tensor(0), 'input_ids': tensor([  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0, 

In [15]:
model = AutoModelForSequenceClassification.from_pretrained("huawei-noah/TinyBERT_General_4L_312D", num_labels=2)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

# Training Arguments

In [17]:
args = TrainingArguments(
    output_dir='./tinybert-sst2', 
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy='epoch',
    learning_rate=2e-5,
    load_best_model_at_end=True,
)

metric = evaluate.load("accuracy")

In [18]:
def compute_metrics(prediction):
    logits, labels = prediction
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [19]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(model=model, args=args, train_dataset=td['train'],eval_dataset=td['validation'],compute_metrics=compute_metrics, data_collator=data_collator)

# Training

In [21]:


trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.259743,0.291254,0.888761
2,0.200486,0.296803,0.891055
3,0.169990,0.311935,0.896789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias'].
There were unexpected keys in the checkp

TrainOutput(global_step=6315, training_loss=0.23202096028452532, metrics={'train_runtime': 325.468, 'train_samples_per_second': 620.789, 'train_steps_per_second': 19.403, 'total_flos': 724287792517632.0, 'train_loss': 0.23202096028452532, 'epoch': 3.0})

In [ ]:

from torch.utils.data import DataLoader

model.eval()
model.to("cuda")

loader = DataLoader(td['validation'], batch_size=32, collate_fn=data_collator)

all_preds = []
all_labels = []

for batch in loader:
    batch = {k: v.to("cuda") for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    preds = np.argmax(outputs.logits.cpu().numpy(), axis=-1)
    all_preds.extend(preds)
    all_labels.extend(batch['labels'].cpu().numpy())

metric.compute(predictions=all_preds, references=all_labels)

{'accuracy': 0.8887614678899083}

## Part 2: Encoder-only Transformers as Feature Extractors

Instead of fine-tuning the full model on a target dataset, it's also possible to simply use the output representations from a BERT-style model as input to a linear classifier and *only* train the classifier (leaving the rest of the pre-trained parameters fixed). 

To achieve this:
1. Pick a pre-trained sentence Transformer.
2. Load the SST-2 dataset and feed the text from each example into the model.
3. Train a linear classifier on the representations.
4. Evaluate performance on the validation set.



In [27]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

st_model = SentenceTransformer('all-MiniLM-L6-v2')

st_model.encode("Test sentence to learn how this works.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\other\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\other\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

array([ 9.04371310e-03,  1.11555539e-01, -4.54469258e-03,  3.89279500e-02,
        8.59768018e-02,  3.10678948e-02,  6.88551515e-02, -7.15284050e-03,
       -2.57497467e-02, -2.21048500e-02,  9.00517330e-02,  1.38311926e-02,
        1.63439736e-02,  2.89288070e-02,  6.64552525e-02,  1.06307277e-02,
        4.77986448e-02,  2.87069567e-02, -3.65213342e-02,  3.79563905e-02,
        7.01531172e-02,  4.91374880e-02,  5.43786176e-02, -1.10665290e-02,
       -1.42056830e-02,  2.68128999e-02, -3.20305489e-02,  5.63309863e-02,
        1.99076068e-02, -3.22729088e-02,  1.26321604e-02,  3.17209139e-02,
        1.02954246e-02,  5.60811497e-02, -3.76783647e-02, -7.27364048e-03,
       -1.31100314e-02,  2.69485600e-02,  1.64898876e-02, -7.13283420e-02,
        2.06631105e-02, -5.17645516e-02, -1.00535145e-02,  9.28030387e-02,
        5.55377174e-03, -8.65538865e-02,  1.97148882e-02, -6.34341910e-02,
        8.80657509e-02, -3.77702853e-03, -6.10447228e-02, -7.67707005e-02,
       -4.39110883e-02, -

In [29]:
train_sentences = dataset['train']['sentence']
val_sentences = dataset['validation']['sentence']

train_labels = dataset['train']['label']
val_labels = dataset['validation']['label']

print("Encoding training set...")
X_train = st_model.encode(train_sentences, show_progress_bar=True)

print("Encoding validation set...")
X_val = st_model.encode(val_sentences, show_progress_bar=True)


print(X_train.shape)

Encoding training set...


Batches:   0%|          | 0/2105 [00:00<?, ?it/s]

Encoding validation set...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [33]:
classifier = LogisticRegression(max_iter=1000)

classifier.fit(X_train, train_labels)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [35]:
val_preds = classifier.predict(X_val)
accuracy = accuracy_score(val_labels, val_preds)
print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.8108


# Comparison

The accuracy drops some when using the fixed sentence transformer. You're going from around 93% on the full BERT model, high 80s in the MiniBert and then low 80s with the LR version. We see huge improvements in compute time and resource needs when we limit what we're putting into the model. The small versions don't require high GPU usage, but suffer in accuracy. 